In [1]:
import GtoTmodel as GtoTmodel
import Circuits as Circuits 
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
circuits = Circuits.CircuitS()


Graph 4['VDD', 'VSS', 'VIN1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 6['VDD', 'VSS', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'R1', 'R1_P', 'R1_N', 'C1', 'C1_P', 'C1_N']
Graph 9['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 14['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'R1', 'R1_P', 'R1_N']
Graph 17['VDD', 'VSS', 'VIN1', 'VOUT1', 'R1', 'R1_P', 'R1_N', 'R2', 'R2_P', 'R2_N', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B']
Graph 20['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B', 'R1', 'R1_P', 'R1_N']
Graph 22['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'PM2', 'PM2_D', 'PM2_G', 'PM2_S', 'PM2_B']
Graph 24['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_

In [4]:
# Assuming circuits has a method or attribute to get the matrix, e.g., circuits.get_matrix()
matrix = circuits.component_lists  # Replace with the actual method or attribute
max_length = max(len(vector) for vector in matrix)
print("Maximum length of vectors in the matrix:", max_length)

Maximum length of vectors in the matrix: 310


In [7]:
circuits.vocab.__len__()  # This should give the number of components

892

In [21]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 892  # Vocabulary size for text


Importing the model

In [22]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [25]:
learning_rate = 0.001
num_epochs = 1000

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Data Load

In [69]:
num_ciruits = 320
graph_dataset =  circuits.graphs
text_dataset = circuits.component_indices

# Convert graph_dataset and text_dataset to numpy arrays
graph_dataset = np.array(graph_dataset, dtype=object)
text_dataset = np.array(text_dataset, dtype=object)

# Convert numpy arrays to PyTorch tensors
graph_dataset = [torch.tensor(graph, dtype=torch.float32) for graph in graph_dataset]
graph_dataset = [torch.cat((torch.nn.functional.pad(graph, (0, max_length - graph.size(0))),torch.tensor([[9]*310]))) for graph in graph_dataset]
text_dataset = [torch.tensor(text+[893], dtype=torch.float32) for text in text_dataset]




In [70]:
graph_dataset[1]
text_dataset[1]

tensor([853., 557., 586., 412., 507., 407., 148., 528., 542.,  84., 398., 623.,
        169., 626., 893.])

In [ ]:
graph_dataset = torch.cat(graph_dataset).to(device)
text_dataset = torch.cat(text_dataset,).to(device)

In [ ]:
from torch.utils.data import DataLoader


train_dataset = torch.utils.data.TensorDataset(graph_dataset, text_dataset)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False,num_workers=4)

In [74]:
graph_dataset.shape, text_dataset.shape

(torch.Size([351461, 310]), torch.Size([351461]))

In [88]:


# Iterate through the DataLoader
for batch_idx, (graph_batch, text_batch) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1}")
    print("Graph Batch Shape:", graph_batch.shape)
    print("Text Batch Shape:", text_batch.shape)

Batch 1
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 2
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 3
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 4
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 5
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 6
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 7
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 8
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 9
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 10
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 11
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.Size([16])
Batch 12
Graph Batch Shape: torch.Size([16, 310])
Text Batch Shape: torch.

Split data in to test and train set

In [94]:
def get_batch(text_dataset, batch_size=4, block_size=16):
    """
    Get a batch of sequences from the text_dataset.

    Args:
        text_dataset (torch.Tensor): The dataset containing text sequences.
        batch_size (int): The number of sequences in the batch.
        block_size (int): The length of each sequence block.

    Returns:
        torch.Tensor: A batch of sequences with shape (batch_size, block_size).
    """
    # Combine graph data and text data for the batch
    start_indices = torch.randint(0, len(graph_dataset) - block_size, (batch_size,))
    batch_graph = torch.stack([graph_dataset[i:i + block_size] for i in start_indices])
    batch_text = torch.stack([text_dataset[i:i + block_size] for i in start_indices])
    # Extract sequences of length block_size starting from the sampled indices

    return batch_text, batch_graph 

# Example usage
batch = get_batch(text_dataset, batch_size=4, block_size=16)
print(batch[1].shape)  # Should print: torch.Size([4, 16])

torch.Size([4, 16, 310])


In [96]:
# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    total_loss = 0  # Initialize total loss for the epoch

    for batch_idx, (graph_batch, text_batch) in enumerate(train_loader):
        # Move data to the appropriate device
        graph_batch, text_batch = graph_batch.to(device), text_batch.to(device)
        text_input = text_batch[:][ :-1]  # Exclude the last token for input
        labels = text_batch[:][ 1:]  # Exclude the first token for labels

        # Forward pass
        outputs = model(graph_batch, text_input)
        outputs = outputs.reshape(-1, text_vocab_size)
        labels = labels.reshape(-1)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Print epoch statistics
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [77]:

graph_train, graph_test, text_train, text_test = graph_dataset[:int(0.8*len(graph_dataset))], graph_dataset[int(0.8*len(graph_dataset)):], text_dataset[:int(0.8*len(text_dataset))], text_dataset[int(0.8*len(text_dataset)):]
graph_train.shape, graph_test.shape, text_train.shape, text_test.shape
train_dataset = torch.utils.data.TensorDataset(graph_train, text_train)


In [ ]:
# Validation phase
model.eval()  # Switch to evaluation mode
# Initialize variables to track metrics
correct = 0
total = 0
all_labels = []
all_predictions = []
cumulative_confidence = 0  # Track prediction confidences

with torch.no_grad():
    for i in range(0, len(graph_test), batch_size):
        # Move inputs and labels to the device (CPU or GPU)
        graph_data = graph_test[i:i+batch_size]
        text = text_test[i:i+batch_size]
        text_input = text[:, :-1]
        labels = text[:, -2:-1].reshape(-1)
        graph_data, text_input, labels = graph_data.to(device), text_input.to(device), labels.to(device)
        
        outputs = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
        outputs = outputs.reshape(-1, text_vocab_size)
        
        # Get the predicted class index (0 or 1)
        probabilities = torch.softmax(outputs, dim=1)
        max_probabilities, predicted = torch.max(probabilities, 1)
        
        # Track prediction confidence
        cumulative_confidence += max_probabilities.mean().item()
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()  # Compare with true labels
        
        # Store all labels and predictions for further metrics calculation
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Calculate accuracy as the percentage of correct predictions
accuracy = 100 * correct / total

# Calculate average prediction confidence
avg_confidence = 100 * cumulative_confidence / (total // batch_size)

# Calculate the confusion matrix
conf_matrix = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = conf_matrix.ravel()

# Print the metrics
print(f'Validation Accuracy: {accuracy:.2f}%')
print(f'Average Prediction Confidence: {avg_confidence:.2f}%')
print(f'True Negatives (Real identified as Real): {tn}')
print(f'False Positives (Real identified as Fake): {fp}')
print(f'False Negatives (Fake identified as Real): {fn}')
print(f'True Positives (Fake identified as Fake): {tp}')
print('Confusion Matrix:')
print(conf_matrix)

# Optional: Calculate additional metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f'\nPrecision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')

Input graph_data shape: torch.Size([2, 4, 10])
graph_encoded shape: torch.Size([2, 4, 16])
graph_encoded permuted shape: torch.Size([4, 2, 16])


RuntimeError: The size of tensor a (8) must match the size of tensor b (2) at non-singleton dimension 0

In [ ]:
import torch.backends.cudnn as cudnn
cudnn.benchmark = True
import torch.backends.cudnn.deterministic = True
import numpy as np
import pandas as pd
import random
import os
import argparse
import time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import networkx as nx
import json
import pickle
import copy
import torch_geometric
import torch_geometric.transforms as T
import torch_geometric.nn as pyg_nn
import torch_geometric.data as pyg_data
import torch_geometric.utils as pyg_utils
import torch_geometric.datasets as pyg_datasets
import torch_geometric.loader as pyg_loader
import torch_geometric.nn as pyg_nn
import torch_geometric.utils as pyg_utils